In [1]:
import shutil
import os
import sys

In [2]:
shutil.which("dggrid")

'/Users/akmoch/dev/build/igeo7_z7_xarray_paper/.pixi/envs/default/bin/dggrid'

In [3]:
os.environ["DGGRID_PATH"]

'/Users/akmoch/dev/build/igeo7_z7_xarray_paper/.pixi/envs/default/bin/dggrid'

In [4]:
dggrid_exec = os.environ["DGGRID_PATH"]

In [5]:
sys.path

['/Users/akmoch/dev/build/igeo7_z7_xarray_paper/scripts/src',
 '/Users/akmoch/dev/build/igeo7_z7_xarray_paper/scripts',
 '/Users/akmoch/dev/build/igeo7_z7_xarray_paper/.pixi/envs/default/lib/python313.zip',
 '/Users/akmoch/dev/build/igeo7_z7_xarray_paper/.pixi/envs/default/lib/python3.13',
 '/Users/akmoch/dev/build/igeo7_z7_xarray_paper/.pixi/envs/default/lib/python3.13/lib-dynload',
 '',
 '/Users/akmoch/dev/build/igeo7_z7_xarray_paper/.pixi/envs/default/lib/python3.13/site-packages',
 '/Users/akmoch/dev/build/igeo7_z7_xarray_paper/xdggs',
 '/Users/akmoch/dev/build/igeo7_z7_xarray_paper/dggrid4py',
 '/Users/akmoch/dev/build/igeo7_z7_xarray_paper/xdggs-dggrid4py']

In [6]:
sys.path.append("/Users/akmoch/dev/build/igeo7_z7_xarray_paper")

In [7]:
sys.path.append("/Users/akmoch/dev/build/igeo7_z7_xarray_paper/src")

In [8]:
from z7py import z7 as Z7
import numpy as np

Z7._INVALID_RAW

np.uint64(18446744073709551615)

In [9]:
import xarray as xr
import numpy as np

# These imports register the xdggs accessor + the IGEO7 grid backend
import xdggs                                          # noqa: F401  -> ds.dggs.*
from xdggs_dggrid4py.index import IGEO7Index          # noqa: F401  -> registers 'igeo7'

import z7_xarray_paper.z7_zarr as z7_zarr


In [10]:
refinement_level = 10

In [11]:
archive = f"../data/working/pori_z7_r{refinement_level}.zarr"

ds = xr.open_zarr(archive, consolidated=False).pipe(xdggs.decode)

In [14]:
ds

<xarray.Dataset> Size: 62kB
Dimensions:         (cell_ids: 3101)
Coordinates:
  * cell_ids        (cell_ids) uint64 25kB 18647993158729727 ... 188036921656...
Data variables:
    elevation       (cell_ids) float32 12kB dask.array<chunksize=(3101,), meta=np.ndarray>
    slope_geodesic  (cell_ids) float32 12kB dask.array<chunksize=(3101,), meta=np.ndarray>
    slope_lookup    (cell_ids) float32 12kB dask.array<chunksize=(3101,), meta=np.ndarray>
Indexes:
    cell_ids  IGEO7Index(level=10)
Attributes:
    clipper_scale_factor:  10000000
    dggs:                  {'compression': 'none', 'coordinate': 'cell_ids', ...
    regridder:             xdggs_dggrid4py.mapblocks_nearestcentroid
    source_crs:            EPSG:3301
    source_path:           /Users/akmoch/dev/build/igeo7_z7_xarray_paper/data...
    zarr_conventions:      [{'description': 'Discrete Global Grid Systems con...

In [15]:
ds.dggs.index

In [16]:
ds['elevation'] = ds['elevation'].compute()

In [17]:
ds['slope_geodesic'] = ds['slope_geodesic'].compute()

In [18]:
ds.dggs.explore()

/Users/akmoch/dev/build/igeo7_z7_xarray_paper/.pixi/envs/default/lib/python3.13/site-packages/lonboard/_geoarrow/ops/reproject.py:40: UserWarning: No CRS exists on data. If no data is shown on the map, double check that your CRS is WGS84.
  warn(


In [14]:
ds2 = z7_zarr.open_dataset("../data/working/pori_z7_r10_ranges.zarr")
ds2

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


<xarray.Dataset> Size: 37kB
Dimensions:    (cell_ids: 3101)
Coordinates:
  * cell_ids   (cell_ids) uint64 25kB dask.array<chunksize=(3101,), meta=np.ndarray>
Data variables:
    elevation  (cell_ids) float32 12kB dask.array<chunksize=(3101,), meta=np.ndarray>
Indexes:
    cell_ids  Z7MonotonicIndex(level=10, R=136, N=3101)
Attributes:
    clipper_scale_factor:  10000000
    dggs:                  {'compression': 'ranges', 'coordinate': 'cell_id_r...
    regridder:             xdggs_dggrid4py.mapblocks_nearestcentroid
    source_crs:            EPSG:3301
    source_path:           /Users/akmoch/dev/build/igeo7_z7_xarray_paper/data...
    zarr_conventions:      [{'description': 'Discrete Global Grid Systems con...

In [ ]:
# works -> MapWithControls
ds2["elevation"].compute().dggs.explore()

In [19]:
ds2.dggs.index

## Phase 3 — slope kernel

Compute slope magnitude (m/m) on the Pori r10 archive using the Z7-native FDA kernel
(`src/z7_xarray_paper/kernels/slope.py`).

Two distance modes:
- **`lookup`** — per-axis weights from the global level-4 distortion table; O(unique level-4 parents); ~1.9% ISEA under-estimate (see AGENT.md §B1)
- **`geodesic`** — per-cell pyproj distances via DGGRID centroids; sub-percent accuracy; O(N×6) calls

Boundary cells (any neighbour outside the AOI) → NaN.

In [15]:
import pandas as pd
import numpy as np
import zarr

from z7_xarray_paper.distance_measures import HexGridDistortionModel
from z7_xarray_paper.kernels import slope as z7_slope

DIST_PARQUET = "../data/working/dist_lookup_level4.parquet"
model = HexGridDistortionModel(
    pd.read_parquet(DIST_PARQUET).to_dict("index")
)
print(f"Distortion model loaded: {len(model.region_weights)} level-4 parents")

Distortion model loaded: 24012 level-4 parents


In [16]:
slope_lookup = z7_slope(ds["elevation"], model, distance_mode="lookup")
slope_geodesic = z7_slope(ds["elevation"], distance_mode="geodesic")
slope_lookup

<xarray.DataArray (cell_ids: 3101)> Size: 25kB
array([       nan, 0.00841775,        nan, ...,        nan, 0.01804639,
              nan], shape=(3101,))
Coordinates:
  * cell_ids  (cell_ids) uint64 25kB 18647993158729727 ... 18803692165660671
Indexes:
    cell_ids  IGEO7Index(level=10)
Attributes:
    units:      m/m
    long_name:  slope magnitude (FDA)

In [30]:
slope_lookup_mono = z7_slope(ds2["elevation"], model, distance_mode="lookup")
slope_geodesic_mono = z7_slope(ds2["elevation"], distance_mode="geodesic")
slope_lookup_mono

<xarray.DataArray (cell_ids: 3101)> Size: 25kB
array([       nan, 0.00841775,        nan, ...,        nan, 0.01804639,
              nan], shape=(3101,))
Coordinates:
  * cell_ids  (cell_ids) uint64 25kB dask.array<chunksize=(3101,), meta=np.ndarray>
Indexes:
    cell_ids  Z7MonotonicIndex(level=10, R=136, N=3101)
Attributes:
    units:      m/m
    long_name:  slope magnitude (FDA)

In [31]:
valid = np.isfinite(slope_lookup_mono.values)
sl = slope_lookup_mono.values[valid]
sg = slope_geodesic_mono.values[valid]

print(f"  {'mode':<12}  {'NaN':>5}  {'median':>9}  {'mean':>9}  {'max':>9}  {'p99':>9}")
print("  " + "-" * 64)
n_nan = int((~valid).sum())
for label, arr in [("lookup", sl), ("geodesic", sg)]:
    print(f"  {label:<12}  {n_nan:>5}  {np.median(arr):>9.5f}  "
          f"{arr.mean():>9.5f}  {arr.max():>9.5f}  {np.percentile(arr, 99):>9.5f}")

delta = np.abs(sl - sg)
print(f"\n  lookup vs geodesic  |Δ| mean = {delta.mean():.6f} m/m  "
      f"rel = {delta.mean() / sg.mean() * 100:.2f}%")

  mode            NaN     median       mean        max        p99
  ----------------------------------------------------------------
  lookup          224    0.01108    0.01335    0.06050    0.04253
  geodesic        224    0.01109    0.01338    0.06170    0.04237

  lookup vs geodesic  |Δ| mean = 0.000188 m/m  rel = 1.41%


In [20]:
archive_path = f"../data/working/pori_z7_r{refinement_level}.zarr"

In [ ]:
zg = zarr.open_group(archive_path, mode="a")
for var_name, da in [("slope_lookup", slope_lookup), ("slope_geodesic", slope_geodesic)]:
    arr = zg.array(var_name, data=da.values.astype(np.float32), overwrite=True)
    arr.attrs.update({
        "_ARRAY_DIMENSIONS": ["cell_ids"],
        "units": "m/m",
        "long_name": da.attrs["long_name"],
    })
print("Archive variables:", sorted(zg.array_keys()))

In [21]:
ds_check = xr.open_zarr(archive_path, consolidated=False).pipe(xdggs.decode)
ds_check

<xarray.Dataset> Size: 62kB
Dimensions:         (cell_ids: 3101)
Coordinates:
  * cell_ids        (cell_ids) uint64 25kB 18647993158729727 ... 188036921656...
Data variables:
    elevation       (cell_ids) float32 12kB dask.array<chunksize=(3101,), meta=np.ndarray>
    slope_geodesic  (cell_ids) float32 12kB dask.array<chunksize=(3101,), meta=np.ndarray>
    slope_lookup    (cell_ids) float32 12kB dask.array<chunksize=(3101,), meta=np.ndarray>
Indexes:
    cell_ids  IGEO7Index(level=10)
Attributes:
    clipper_scale_factor:  10000000
    dggs:                  {'compression': 'none', 'coordinate': 'cell_ids', ...
    regridder:             xdggs_dggrid4py.mapblocks_nearestcentroid
    source_crs:            EPSG:3301
    source_path:           /Users/akmoch/dev/build/igeo7_z7_xarray_paper/data...
    zarr_conventions:      [{'description': 'Discrete Global Grid Systems con...

In [22]:
ds_check["slope_lookup"].compute().dggs.explore()

/Users/akmoch/dev/build/igeo7_z7_xarray_paper/.pixi/envs/default/lib/python3.13/site-packages/lonboard/_geoarrow/ops/reproject.py:40: UserWarning: No CRS exists on data. If no data is shown on the map, double check that your CRS is WGS84.
  warn(


In [ ]:
# Same write for the ranges archive — Z7MonotonicIndex is preserved end-to-end
ranges_path = f"../data/working/pori_z7_r{refinement_level}_ranges.zarr"
s_ranges = z7_slope(ds2["elevation"], model, distance_mode="lookup")

zg2 = zarr.open_group(ranges_path, mode="a")
arr2 = zg2.array("slope_lookup", data=s_ranges.values.astype(np.float32), overwrite=True)
arr2.attrs.update({"_ARRAY_DIMENSIONS": ["cell_ids"], "units": "m/m",
                   "long_name": s_ranges.attrs["long_name"]})

import z7_xarray_paper.z7_zarr as z7_zarr
ds_check2 = z7_zarr.open_dataset(ranges_path)
ds_check2

In [ ]:
ds_check2["slope_lookup"].dggs.explore()

## Phase 3 HaloChunk + map_blocks for slope

In [12]:
import pandas as pd
import numpy as np
import zarr

from z7_xarray_paper.distance_measures import HexGridDistortionModel
from z7_xarray_paper.kernels import slope as z7_slope, slope_blocked as z7_slope_blocked

DIST_PARQUET = "../data/working/dist_lookup_level4.parquet"
model = HexGridDistortionModel(
    pd.read_parquet(DIST_PARQUET).to_dict("index")
)
print(f"Distortion model loaded: {len(model.region_weights)} level-4 parents")

Distortion model loaded: 24012 level-4 parents


In [13]:
ds = z7_zarr.open_dataset("../data/working/pori_z7_r10_ranges.zarr").chunk({"cell_ids": 117649})
ds

<xarray.Dataset> Size: 50kB
Dimensions:       (cell_ids: 3101)
Coordinates:
  * cell_ids      (cell_ids) uint64 25kB dask.array<chunksize=(3101,), meta=np.ndarray>
Data variables:
    elevation     (cell_ids) float32 12kB dask.array<chunksize=(3101,), meta=np.ndarray>
    slope_lookup  (cell_ids) float32 12kB dask.array<chunksize=(3101,), meta=np.ndarray>
Indexes:
    cell_ids  Z7MonotonicIndex(level=10, R=136, N=3101)
Attributes:
    clipper_scale_factor:  10000000
    dggs:                  {'compression': 'ranges', 'coordinate': 'cell_id_r...
    regridder:             xdggs_dggrid4py.mapblocks_nearestcentroid
    source_crs:            EPSG:3301
    source_path:           /Users/akmoch/dev/build/igeo7_z7_xarray_paper/data...
    zarr_conventions:      [{'description': 'Discrete Global Grid Systems con...

In [14]:
ds.dggs.index

In [15]:
slope_da = z7_slope_blocked(ds["elevation"], model, distance_mode="lookup")

In [16]:
slope_da.compute()

<xarray.DataArray 'elevation' (cell_ids: 3101)> Size: 25kB
array([       nan, 0.00841775,        nan, ...,        nan, 0.01804639,
              nan], shape=(3101,))
Coordinates:
  * cell_ids  (cell_ids) uint64 25kB 18647993158729727 ... 18803692165660671
Indexes:
    cell_ids  Z7MonotonicIndex(level=10, R=136, N=3101)
Attributes:
    units:      m/m
    long_name:  slope magnitude (FDA)

In [17]:
ds["slope"] = slope_da

In [19]:
ds["slope"].compute().dggs.explore()

/Users/akmoch/dev/build/igeo7_z7_xarray_paper/.pixi/envs/default/lib/python3.13/site-packages/lonboard/_geoarrow/ops/reproject.py:40: UserWarning: No CRS exists on data. If no data is shown on the map, double check that your CRS is WGS84.
  warn(
